# Worked Example: Time-Frequency Reward Contrast

## Goal
Morlet TFR with cross-event trialwise baseline normalization. Load saved **feedback** and **baseline** epoch files from chapter 07; Morlet runs on buffered voltage epochs; buffers are cropped on the TFR axis before z-scoring and plotting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from LFPAnalysis import load_lfp, run_analysis
from LFPAnalysis.config import AnalysisConfig, LoadConfig, TfrConfig

chan = 'racas1-racas2'
freqs = np.arange(4, 30, 4).tolist()

task_epochs = load_lfp(
    LoadConfig(path=Path('../../data/sample_feedback_start-epo.fif'), file_format='mne', preload=True)
)
baseline_epochs = load_lfp(
    LoadConfig(path=Path('../../data/sample_baseline_start-epo.fif'), file_format='mne', preload=True)
)

print('Voltage epoch spans (buffers kept for Morlet):')
print('  feedback_start:', task_epochs.times[0], task_epochs.times[-1])
print('  baseline_start:', baseline_epochs.times[0], baseline_epochs.times[-1])

result = run_analysis(
    task_epochs,
    AnalysisConfig(
        tfr=TfrConfig(
            enabled=True,
            method='morlet',
            freqs=freqs,
            n_cycles=3.0,
            baseline_mode='trialwise',
            apply_baseline=True,
            crop_tmin=-0.5,
            crop_tmax=1.5,
            baseline_crop_tmin=-0.5,
            baseline_crop_tmax=0.0,
        )
    ),
    baseline_epochs=baseline_epochs,
)

power = result.tfr['power'].copy().pick([chan])
print('TFR time span after crop:', power.times[0], power.times[-1])

reward_ix = np.where(power.metadata['reward'].to_numpy() == 1)[0]
loss_ix = np.where(power.metadata['reward'].to_numpy() == 0)[0]
reward_map = np.nanmean(power.data[reward_ix, 0], axis=0)
loss_map = np.nanmean(power.data[loss_ix, 0], axis=0)
diff = reward_map - loss_map
print('TFR shape:', power.data.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
freq_arr = np.asarray(freqs)
for ax, data, title in zip(
    axes,
    [reward_map, loss_map, diff],
    ['reward', 'no reward', 'difference'],
):
    im = ax.imshow(
        data,
        aspect='auto',
        origin='lower',
        extent=[power.times[0], power.times[-1], freq_arr[0], freq_arr[-1]],
        cmap='RdBu_r',
    )
    ax.axvline(0, color='k', ls='--', lw=0.8)
    ax.set(xlabel='Time (s)', title=title)
axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(im, ax=axes, shrink=0.8, label='Trialwise z-scored power')
fig.tight_layout()
plt.show()

## Saving results

See chapter 15 (`15_saving_and_organizing_results`) for the recommended `results/` layout.

In [ ]:
# Uncomment to save. See chapter 15 for the recommended results/ layout.
# out = Path('../../results/worked-examples')
# out.mkdir(parents=True, exist_ok=True)
# power.save(out / 'feedback_cross_event_trialwise-tfr.h5', overwrite=True)
# Reload with: mne.time_frequency.read_tfrs(out / 'feedback_cross_event_trialwise-tfr.h5')

## Next step

Chapter 10 (`10_first_connectivity_and_surrogates`) covers connectivity.